# Build the SqueakView YOLO26 Pose Engine

This notebook is self-contained. It reads the dataset YAML, exports the pose checkpoint directly with Ultralytics, converts the Ultralytics engine into the raw TensorRT plan expected by DeepStream, and writes the model package.

In [ ]:
from pathlib import Path

import hashlib
import json
import os
import shutil

import onnx
import tensorrt as trt
import torch
import ultralytics
import yaml
from ultralytics import YOLO

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name in {"build_engine", "build-engine"}:
    WORKSPACE = WORKSPACE.parent

SOURCE_MODEL = WORKSPACE / "build_me/mousehouse_model/mousehouse_best.pt"
DATA_YAML = WORKSPACE / "build_me/mousehouse_model/mousehouse_best.yaml"
PARSER_LIBRARY = WORKSPACE / "native/nvdsinfer_custom_impl_yolo/libnvdsinfer_custom_impl_Yolo.so"
MODEL_NAME = "mousehouse"
PACKAGE_DIR = WORKSPACE / "models" / MODEL_NAME

PRECISION = "fp16"
BATCH_SIZE = 1
IMAGE_SIZE = 640
DEVICE = 0
CONFIDENCE_THRESHOLD = 0.25
KEYPOINT_THRESHOLD = 0.50
OVERWRITE_EXISTING = False

assert SOURCE_MODEL.is_file(), SOURCE_MODEL
assert DATA_YAML.is_file(), DATA_YAML
assert PARSER_LIBRARY.is_file(), PARSER_LIBRARY
assert torch.cuda.is_available(), "CUDA is required for TensorRT export"

print(f"Ultralytics {ultralytics.__version__}")
print(f"PyTorch {torch.__version__}")
print(f"TensorRT {trt.__version__}")
print(f"GPU {torch.cuda.get_device_name(DEVICE)}")


In [ ]:
# Derive the complete runtime label/index contract from the YAML and checkpoint.
data_config = yaml.safe_load(DATA_YAML.read_text()) or {}
model = YOLO(str(SOURCE_MODEL))
head = model.model.model[-1]

def ordered_labels(value):
    if isinstance(value, dict):
        def sort_key(key):
            return (0, int(key)) if str(key).isdigit() else (1, str(key))
        return [str(value[key]) for key in sorted(value, key=sort_key)]
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value]
    return []

def class_value(mapping, class_id, class_name):
    if not isinstance(mapping, dict):
        return None
    for key in (class_id, str(class_id), class_name):
        if key in mapping:
            return mapping[key]
    return None

checkpoint_classes = [str(model.names[index]) for index in sorted(model.names)]
class_names = ordered_labels(data_config.get("names")) or checkpoint_classes
checkpoint_kpt_shape = [int(value) for value in head.kpt_shape]
yaml_kpt_shape = data_config.get("kpt_shape")
keypoint_count, keypoint_dims = map(int, yaml_kpt_shape or checkpoint_kpt_shape)
assert model.task == "pose", f"Expected pose, found {model.task}"
assert bool(head.end2end), "Expected a YOLO26 end-to-end checkpoint"
assert checkpoint_classes == class_names, (checkpoint_classes, class_names)
assert checkpoint_kpt_shape == [keypoint_count, keypoint_dims]
assert keypoint_dims == 3, "Pose keypoints must use x/y/confidence"

def global_keypoint_names():
    sources = [
        ("yaml.kp_names", data_config.get("kp_names")),
        ("yaml.keypoint_names", data_config.get("keypoint_names")),
        ("yaml.kpt_names", data_config.get("kpt_names")),
        ("checkpoint.kpt_names", getattr(model, "kpt_names", None)),
    ]
    for source_name, value in sources:
        candidates = []
        if isinstance(value, (list, tuple)):
            candidates = [[str(item) for item in value]]
        elif isinstance(value, dict):
            for class_id, class_name in enumerate(class_names):
                candidate = class_value(value, class_id, class_name)
                if isinstance(candidate, (list, tuple)):
                    candidates.append([str(item) for item in candidate])
        valid = [candidate for candidate in candidates if len(candidate) == keypoint_count]
        if valid and all(candidate == valid[0] for candidate in valid):
            # Numeric checkpoint labels are placeholders; keep searching for YAML names.
            if source_name.startswith("checkpoint") and all(name.isdigit() for name in valid[0]):
                continue
            return valid[0], source_name
    return [f"keypoint_{index}" for index in range(keypoint_count)], "generated"

keypoint_names, keypoint_name_source = global_keypoint_names()
assert len(keypoint_names) == keypoint_count
assert len(set(keypoint_names)) == keypoint_count, "Keypoint names must be unique"
pose_classes = [
    {
        "id": class_id,
        "name": class_name,
        "threshold": CONFIDENCE_THRESHOLD,
        "track": class_id == 0,
        "keypoint_indices": list(range(keypoint_count)),
    }
    for class_id, class_name in enumerate(class_names)
]

EXPECTED_OUTPUT = [BATCH_SIZE, 300, 6 + keypoint_count * keypoint_dims]
print("classes:", class_names)
print(f"keypoints ({keypoint_name_source}):", keypoint_names)
print("class policies:", pose_classes)
print("expected output:", EXPECTED_OUTPUT)


In [ ]:
# Export directly with Ultralytics. DATA_YAML is passed as data=.
weights_dir = PACKAGE_DIR / "weights"
onnx_dir = PACKAGE_DIR / "onnx"
engines_dir = PACKAGE_DIR / "engines"
labels_dir = PACKAGE_DIR / "labels"
configs_dir = PACKAGE_DIR / "configs"
validation_dir = PACKAGE_DIR / "validation"
for directory in (weights_dir, onnx_dir, engines_dir, labels_dir, configs_dir, validation_dir):
    directory.mkdir(parents=True, exist_ok=True)

artifact_stem = f"{SOURCE_MODEL.stem}_{PRECISION}_b{BATCH_SIZE}"
packaged_model = weights_dir / SOURCE_MODEL.name
onnx_path = onnx_dir / f"{artifact_stem}.onnx"
engine_path = engines_dir / f"{artifact_stem}.engine"
if not OVERWRITE_EXISTING and any(path.exists() for path in (onnx_path, engine_path)):
    raise FileExistsError(f"Package artifacts already exist under {PACKAGE_DIR}")

shutil.copy2(SOURCE_MODEL, packaged_model)
export_model = YOLO(str(packaged_model))
ultralytics_engine = Path(export_model.export(
    format="engine",
    device=DEVICE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    dynamic=False,
    quantize=16 if PRECISION == "fp16" else 32,
    simplify=True,
    end2end=True,
    data=str(DATA_YAML),
)).resolve()
exported_onnx = packaged_model.with_suffix(".onnx")
assert ultralytics_engine.is_file()
assert exported_onnx.is_file()

if onnx_path.exists():
    onnx_path.unlink()
shutil.move(str(exported_onnx), onnx_path)

# Ultralytics prefixes its plan with a 4-byte JSON length and JSON metadata.
# DeepStream needs only the raw TensorRT plan that follows it.
with ultralytics_engine.open("rb") as source:
    metadata_size = int.from_bytes(source.read(4), "little", signed=True)
    assert 0 < metadata_size < 16 * 1024 * 1024
    engine_metadata = json.loads(source.read(metadata_size))
    raw_plan = source.read()
assert raw_plan, "The exported TensorRT plan is empty"
engine_path.write_bytes(raw_plan)
ultralytics_engine.unlink()

# Validate ONNX structure and the raw TensorRT plan before packaging.
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
def tensor_shape(value_info):
    return [int(dim.dim_value) if dim.dim_value else dim.dim_param for dim in value_info.type.tensor_type.shape.dim]
input_shape = tensor_shape(onnx_model.graph.input[0])
output_shapes = [tensor_shape(output) for output in onnx_model.graph.output]
assert input_shape == [BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE], input_shape
assert output_shapes == [EXPECTED_OUTPUT], output_shapes
runtime = trt.Runtime(trt.Logger(trt.Logger.ERROR))
assert runtime.deserialize_cuda_engine(raw_plan) is not None
print("ONNX:", onnx_path)
print("DeepStream engine:", engine_path)

In [ ]:
# Write the complete schema-v2 DeepStream/SqueakView model package.
classes_path = labels_dir / "classes.txt"
keypoints_path = labels_dir / "labels.txt"
classes_path.write_text("\n".join(class_names) + "\n")
keypoints_path.write_text("\n".join(keypoint_names) + "\n")

config_path = configs_dir / f"{MODEL_NAME}.txt"
parser_path = os.path.relpath(PARSER_LIBRARY, configs_dir).replace(os.sep, "/")
network_mode = 2 if PRECISION == "fp16" else 0
config_path.write_text(f"""[property]
gpu-id=0
net-scale-factor=0.00392156862745098
model-color-format=0
onnx-file=../onnx/{onnx_path.name}
model-engine-file=../engines/{engine_path.name}
network-mode={network_mode}
network-type=0
infer-dims=3;{IMAGE_SIZE};{IMAGE_SIZE}
batch-size={BATCH_SIZE}
output-tensor-meta=1
num-detected-classes={len(class_names)}
labelfile-path=../labels/classes.txt
parse-bbox-func-name=NvDsInferParseYolo26Pose
custom-lib-path={parser_path}
cluster-mode=4
maintain-aspect-ratio=1
symmetric-padding=1
gie-unique-id=1
interval=0
process-mode=1

[class-attrs-all]
pre-cluster-threshold={CONFIDENCE_THRESHOLD}
topk=300
""")

output_layer = onnx_model.graph.output[0].name
pose_schema = {
    "schema_version": 2,
    "task": "pose",
    "postprocess": "pyservicemaker_yolo26_pose_v1",
    "output_layer": output_layer,
    "input_width": IMAGE_SIZE,
    "input_height": IMAGE_SIZE,
    "letterbox": "symmetric",
    "end2end": True,
    "keypoint_labels_path": "../labels/labels.txt",
    "keypoint_count": keypoint_count,
    "keypoint_dims": keypoint_dims,
    "keypoint_threshold": KEYPOINT_THRESHOLD,
    "classes": pose_classes,
}
pose_path = configs_dir / f"{MODEL_NAME}.pose.json"
pose_path.write_text(json.dumps(pose_schema, indent=2) + "\n")

try:
    data_reference = DATA_YAML.resolve().relative_to(WORKSPACE).as_posix()
except ValueError:
    data_reference = DATA_YAML.name
dataset_sha256 = hashlib.sha256(DATA_YAML.read_bytes()).hexdigest()
manifest_path = PACKAGE_DIR / "model.yaml"
manifest_path.write_text(yaml.safe_dump({
    "schema_version": 2,
    "name": MODEL_NAME,
    "framework": "yolo26",
    "task": "pose",
    "precision": PRECISION,
    "batch_size": BATCH_SIZE,
    "classes": class_names,
    "keypoints": keypoint_names,
    "export": {
        "builder": "ultralytics", "data": data_reference,
        "data_sha256": dataset_sha256, "end2end": True,
    },
}, sort_keys=False))

report_path = validation_dir / "import_report.json"
report_path.write_text(json.dumps({
    "onnx_input_shape": input_shape,
    "onnx_output_shapes": output_shapes,
    "output_layer": output_layer,
    "ultralytics_engine_metadata": engine_metadata,
    "checks": {
        "onnx": True, "raw_engine": True, "yaml_labels": True, "schema_v2": True,
    },
}, indent=2) + "\n")

required = [packaged_model, onnx_path, engine_path, classes_path, keypoints_path, config_path, pose_path, manifest_path, report_path]
assert all(path.is_file() for path in required)
assert pose_schema["schema_version"] == 2
assert pose_schema["classes"] == pose_classes
print("Package ready:", PACKAGE_DIR)
print("Select config:", config_path)
